# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeref538/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: scoring** (supervised).

My lane is Refresh / Content Opportunity Scoring. The decision is "which pages get the ~50 refresh slots we can ship this cycle," so the honest answer sounds like classification ("is this page declining, yes/no?") but a yes/no flag can't fill 50 ordered slots — as Mirza put it in the Week-2 session, we need an **order**, so we frame it as **scoring**: output a risk score per page and sort into a ranked queue. It's supervised because each row can carry an answer (declining or not).

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**What I really want** is "should this page get a refresh slot?" — but I can't measure that directly (nobody labeled it). So I predict a **proxy** I *can* measure: whether the page is declining.

- **Starter proxy (this notebook):** `is_declining = trend_direction == "down"`. Honest about its weakness — `trend_direction` is *derived* from `trend_pct` over the same trailing window, so it's a defined/derived label, not a clean future outcome. That also means `trend_direction` and `trend_pct` are **banned as features** (they'd leak the answer).
- **Where the real label should come from (capstone):** an **observed future outcome**, not a current bucket — features from the prior 90 days → did impressions decline over the *next* 30 days, built from the warehouse daily table. Prior-window features, next-window label, no overlap.

So: proxy now (to get the pipeline running honestly), observed future label later.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.**

Of the top 50 pages my score sends to the front of the refresh queue, how many are actually declining? I pick this because it matches the real decision — the team can only ship ~50 refreshes a cycle, so only the *top* of the ranking is ever acted on. Overall accuracy would reward being right about the thousands of pages nobody will ever touch; Precision@50 only rewards being right where it costs money.

**What "good" means:** beat the transparent rule baseline. In the starter pipeline that baseline scored **0.240** (≈12 of 50 right) and the random forest hit **0.740** (≈37 of 50) under client-holdout — the 24→74 gap from the session. My bar: clear the 0.240 baseline under an honest client-holdout split, and report Precision@20 too since a reviewer may only check the top 20.

**The action this output supports:** the ranked queue goes to a content editor, who opens the top pages in order and takes one concrete action per page — **refresh** (rewrite/update), **expand**, **prune**, or **monitor** — each shown with a reason code so the editor sees *why* it was flagged. The model prioritizes the 50 slots; the human decides what to do with each.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import os
import pandas as pd

# work/notebooks/ -> repo root (works locally; Colab badge clones the repo)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# One row = one content page (for one client). That's my unit of analysis.
print(f"Grain check: {len(df):,} rows, {df['content_id'].nunique():,} unique content_ids "
      f"-> {'one row per page' if len(df) == df['content_id'].nunique() else 'DUPLICATES!'}")
print(f"Clients: {df['client_id'].nunique()} (I split train/test by client so a client's "
      f"pages never appear in both)\n")

# The proxy label lives on each row.
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"Proxy label rate: {df['is_declining'].mean():.1%} of pages are declining\n")

# Show the unit: a few pages with the signals I'd score on + the label.
cols = ["content_id", "client_id", "impressions_90d", "avg_position", "ctr",
        "days_since_last_update", "is_declining"]
df[cols].head(5)

Grain check: 30,000 rows, 30,000 unique content_ids -> one row per page
Clients: 32 (I split train/test by client so a client's pages never appear in both)

Proxy label rate: 54.2% of pages are declining



,content_id,client_id,impressions_90d,avg_position,ctr,days_since_last_update,is_declining
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,20,1
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,25,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,20,1
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,22,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,14,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed `if`-statement can catch the *obvious* case (Mirza's "if page older than a year and traffic dropped, flag it"), but it runs out of signal fast here, for three reasons I already saw in Week 1:

1. **The signal is spread across many tangled features.** Decline isn't one threshold — it's position × freshness × impressions × CTR × engagement interacting. A hand rule can weigh two of those; a model weighs all of them at once.
2. **Intuition is sometimes backwards.** In this data, declining pages hold *better* median positions than growing ones (11.3 vs 15.3) — so a gut rule like "fix the worst-ranked first" aims at the wrong pages. A model learns the real shape instead of my assumption.
3. **The rule can't even fill the queue.** "Stale AND visible" flagged only ~17 pages while ~16,000 pages are declining — a rule filters, but scoring is what *orders* the rest into 50 slots.

This is exactly the "real but too messy for hand-rules" case where ML earns its place — and the honest test still decides it: if the score can't beat the 0.240 baseline under client-holdout, the rule wins and that's the finding.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.